# 4. Scraper performance

*Author: Phạm Tấn Nhật Thịnh · code: `src/04_monitor.ipynb`, `src/05_run_pipeline.ipynb`*

This chapter reports the running time (4.2) and the memory and disk footprint (4.3) of the crawl of **all
20,000 assigned IDs** on a CPU-only Google Colab runtime (Q&A M1 #9, #16), summarises the metrics the
statement asks for in one table (4.4) and discusses what limits the speed (4.5). As in the previous chapter,
every number, table and interactive figure is computed from the crawl's logs when the book is built.

## 4.1 Testbed and measurement

| | |
|---|---|
| **Machine** | Google Colab, CPU-only runtime (free tier), as required by the statement |
| **Storage** | corpus, state and logs on Google Drive (`MyDrive/arxiv-project`); downloaded archives on the runtime's local disk (`/tmp`) |
| **Code** | `src/05_run_pipeline.ipynb`, the same notebook for every session |
| **Network policy** | one connection to arXiv, 3 s between requests; one connection to Semantic Scholar |

The crawl did not fit in one 12-hour free Colab session, so it ran as a sequence of **30 resumable sessions**
of the same notebook, with every per-session log preserved in `benchmark_sessions.jsonl` and summed in
`benchmark.json` (Q&A M1 #15). All 3 s rate-limit delays, HTTP 429/5xx retries, backoff waits and resume
overhead are **included** in the reported wall time and per-paper averages (Q&A M1 #13):

| Metric | How it is measured | Where it is saved |
|---|---|---|
| wall time | `time.perf_counter()` around the session and around each stage (delays and retries included) | `logs/sessions/*.json` → `benchmark_sessions.jsonl` |
| time per version | from the moment the downloader starts on the version (including the wait for its request slot) until its files are on Drive | receipt of the version (`elapsed_seconds`) |
| entry discovery time | time spent in arXiv API calls, added up over the runs | `state/discovery_manifest.json` |
| RAM | resident set size of the notebook process, sampled every 5 s by a monitor thread with `psutil` | `logs/ram_timeseries.csv`, per-session peak and average |
| disk | bytes of the files kept (from the receipts), size of the project folder, peak of the archives waiting in `/tmp` | session records |

Three caveats apply to all numbers below. A session that Colab killed could not write its final record:
it is listed as *interrupted*, and its wall time is the time of its last periodic save, so it is a lower
bound. RAM is sampled, so a peak shorter than 5 s can be missed. Sizes use binary units (1 MB = 2²⁰ bytes,
1 GB = 2³⁰ bytes).

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import report_data as rd

rd.check()
bench, perf = rd.benchmark(), rd.performance()
sessions, versions, events, papers = rd.sessions(), rd.versions(), rd.events(), rd.papers()
t0 = sessions["start"].min()


def hours(seconds):
    return seconds / 3600


def hms(seconds):
    seconds = int(round(seconds))
    return f"{seconds // 3600:d} h {seconds % 3600 // 60:02d} min"

## 4.2 Running time

In [ ]:
ordered = versions.sort_values("finished_at")
gaps = ordered["finished_at"].diff().dt.total_seconds().fillna(0)
PAUSE = 600   # a gap of more than 10 minutes between two receipts is a pause, not crawling
active_seconds = gaps[gaps <= PAUSE].sum()
main_end = ordered["finished_at"].iloc[int(len(ordered) * 0.999)]   # end of the main crawl, before the retry session
calendar_seconds = (main_end - t0).total_seconds()
stage = sessions[["discovery_seconds", "download_seconds", "references_seconds"]].sum()
done = papers[(papers["versions"] > 0) & (papers["outcome"] != "incomplete")]

rows = [
    ("Sessions", f"{len(sessions)} ({int(sessions['complete'].sum())} completed, "
                 f"{int((~sessions['complete']).sum())} interrupted)"),
    ("Calendar time, first session start → end of the main crawl", hms(calendar_seconds)),
    ("Total wall time, sum over the sessions (delays & retries included)",
     f"{hms(bench['wall_seconds_total'])}  ({bench['wall_seconds_total']:,.0f} s)"),
    ("Active download time estimated from the receipts", hms(active_seconds)),
    ("Entry discovery time, all API calls", f"{bench['discovery_seconds'] / 60:.1f} min  ({bench['discovery_seconds']:,.0f} s)"),
    ("Download stage, sum over the completed sessions", hms(stage["download_seconds"])),
    ("Reference stage, sum over the sessions (runs in parallel with the downloads)", hms(stage["references_seconds"])),
    ("Average time per paper, end to end (total wall time / papers)", f"{bench['avg_seconds_per_paper_end_to_end']:.2f} s"),
    ("Time per paper, download and processing: median", f"{done['seconds'].median():.2f} s"),
    ("Time per paper, download and processing: mean", f"{done['seconds'].mean():.2f} s"),
    ("Time per version: median / mean", f"{versions['seconds'].median():.2f} s / {versions['seconds'].mean():.2f} s"),
]
rd.table(pd.DataFrame(rows, columns=["Running time", "Value"]), "Table 4.1. Running time of the crawl.")
rd.md(
    f"The crawl took **{hms(bench['wall_seconds_total'])} of recorded wall time** over {len(sessions)} sessions "
    f"between {t0:%d %b} and {main_end:%d %b %Y}; a last short session on {sessions['start'].max():%d %b} only "
    f"retried the versions that had failed. Divided by the {bench['papers_on_arxiv']:,} papers this is "
    f"**{bench['avg_seconds_per_paper_end_to_end']:.2f} s per paper end to end**, with discovery, references, "
    f"3 s waits, retries and resume checks all included.\n\n"
    f"The recorded wall time is not exact: interrupted sessions are cut at their last save, and "
    f"{int((~sessions['complete']).sum())} sessions were interrupted. As a cross-check, the receipts give an "
    f"independent estimate: adding up the time between consecutive receipts, and treating gaps of more than "
    f"{PAUSE // 60} minutes as pauses, the downloader was active for about **{hms(active_seconds)}**. "
    f"**Entry discovery took {bench['discovery_seconds'] / 60:.1f} minutes** for all 20,000 IDs and every "
    f"version, because the API is asked for 100 IDs per call; the few seconds of discovery in the later sessions "
    f"are the check that the saved manifest is complete.")

In [ ]:
x_all = hours((ordered["finished_at"] - t0).dt.total_seconds()).to_numpy()
y_all = np.arange(1, len(ordered) + 1)
max_h = float(hours((main_end - t0).total_seconds()) * 1.03)
step = max(1, len(x_all) // 800)

fig = go.Figure()
for row in sessions.itertuples():
    x0 = float(hours((row.start - t0).total_seconds()))
    x1 = float(hours((row.end - t0).total_seconds()))
    if x0 > max_h:
        continue
    fig.add_vrect(
        x0=x0, x1=min(x1, max_h),
        fillcolor=rd.BLUE if row.complete else rd.ORANGE,
        opacity=0.14, line_width=0,
    )
fig.add_trace(go.Scatter(
    x=x_all[::step], y=y_all[::step], mode="lines",
    name="versions with a receipt", line=dict(color=rd.BLUE, width=2.2),
    hovertemplate="Hour %{x:.1f}<br><b>%{y:,}</b> versions processed<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode="markers",
    marker=dict(size=10, symbol="square", color=rd.BLUE, opacity=0.35),
    name="completed session",
))
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode="markers",
    marker=dict(size=10, symbol="square", color=rd.ORANGE, opacity=0.35),
    name="interrupted session (recorded part)",
))
fig.add_hline(
    y=bench["versions_total"], line_dash="dash", line_color=rd.MUTED, line_width=1.2,
    annotation_text=f"{bench['versions_total']:,} versions to process", annotation_position="bottom left",
)
fig.update_layout(
    title="Progress of the crawl",
    legend=dict(orientation="h", yanchor="top", y=-0.22, xanchor="left", x=0.0),
)
fig.update_xaxes(title_text=f"Hours since the first session started ({t0:%d %b %Y %H:%M} UTC)", range=[0, max_h])
fig.update_yaxes(title_text="Versions processed", range=[0, bench["versions_total"] * 1.06])
rd.show(fig, height=370, margin=dict(l=60, r=25, t=55, b=75))
rd.figure_caption("Figure 4.1. Cumulative number of processed versions over calendar time, from the time stamps of "
                  "the receipts. The shaded bands are the recorded sessions; progress outside a band belongs to an "
                  "interrupted session whose end was not recorded. The last retry session is not shown.")

inside = np.zeros(len(versions), dtype=bool)
for row in sessions.itertuples():
    inside |= ((versions["finished_at"] >= row.start) & (versions["finished_at"] <= row.end)).to_numpy()
rd.md(
    f"{(~inside).sum():,} receipts ({(~inside).mean():.1%}) carry a time stamp outside every recorded session: "
    f"they were written by sessions that were killed afterwards, which is the part of the running time the "
    f"session records miss. The flat stretches of the curve are the gaps between sessions and the periods in "
    f"which arXiv refused the runtime's IP (section 4.5).")

In [ ]:
table = pd.DataFrame({
    "Start (UTC)": sessions["start"].dt.strftime("%m-%d %H:%M"),
    "Status": np.where(sessions["complete"], "completed", "interrupted"),
    "Wall time (h)": hours(sessions["wall_seconds"]),
    "Download (h)": hours(sessions["download_seconds"]),
    "References (min)": sessions["references_seconds"] / 60,
    "Versions": [int(((versions["finished_at"] >= r.start) & (versions["finished_at"] <= r.end)).sum())
                 for r in sessions.itertuples()],
    "Peak RAM (MB)": sessions["ram_peak_mb"],
    "Avg RAM (MB)": sessions["ram_avg_mb"],
    "Project on disk (MB)": sessions["project_bytes"] / rd.MB,
    "Peak in /tmp (MB)": sessions["local_temp_peak_bytes"] / rd.MB,
})
table.insert(0, "Session", range(1, len(table) + 1))
rd.table(table, "Table 4.2. The Colab sessions of the crawl. 'Versions' counts the receipts written inside the "
                "recorded window of the session.",
         {"Wall time (h)": "{:.2f}", "Download (h)": "{:.2f}", "References (min)": "{:.1f}", "Peak RAM (MB)": "{:.0f}",
          "Avg RAM (MB)": "{:.0f}", "Project on disk (MB)": "{:,.0f}", "Peak in /tmp (MB)": "{:.0f}"})
longest = sessions.loc[sessions["wall_seconds"].idxmax()]
long_sessions = sessions[sessions["wall_seconds"] >= 3 * 3600]
rd.md(
    f"The work is concentrated in a few long sessions: the {len(long_sessions)} sessions of three hours or more "
    f"hold {long_sessions['wall_seconds'].sum() / sessions['wall_seconds'].sum():.0%} of the wall time, and the "
    f"longest ran for {hours(longest['wall_seconds']):.1f} h, close to the lifetime of a free Colab runtime. The "
    f"many short sessions are restarts, after a disconnect or on a fresh runtime.")

### Time per version and per paper

The time of a version runs from the moment the downloader starts on it, including the wait for its request
slot, until its files are on Drive. Unpacking and the writes to Drive happen in two worker threads while the
main thread is already waiting for the next slot, so they do not add to the pace. A paper's time is the sum
over its versions.

In [ ]:
seconds = versions["seconds"]
cap = 30
bins = np.arange(0, cap + 0.5, 0.5)
h_sec, edges = np.histogram(seconds.clip(upper=cap), bins=bins)
sample = versions.sample(min(len(versions), 4000), random_state=0)

fig = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.12,
    subplot_titles=("Time per version", f"Time vs archive size ({len(sample):,}-version sample)"),
)
fig.add_trace(
    go.Bar(
        x=0.5 * (edges[:-1] + edges[1:]), y=h_sec, width=0.45, marker_color=rd.BLUE, showlegend=False,
        customdata=np.stack([edges[:-1], edges[1:]], axis=-1),
        hovertemplate="%{customdata[0]:.1f}–%{customdata[1]:.1f} s<br><b>%{y:,}</b> versions<extra></extra>",
    ),
    row=1, col=1,
)
fig.add_vline(
    x=rd.ARXIV_DELAY, line_dash="dash", line_color=rd.INK, line_width=1.3,
    annotation_text=f"{rd.ARXIV_DELAY:g} s interval", annotation_position="top right",
    row=1, col=1,
)
fig.add_trace(
    go.Scattergl(
        x=sample["downloaded_bytes"].clip(lower=1024) / rd.MB,
        y=sample["seconds"],
        mode="markers",
        marker=dict(size=4, color=rd.BLUE, opacity=0.35),
        showlegend=False,
        customdata=np.stack([sample["id"], sample["version"]], axis=-1),
        hovertemplate="<b>%{customdata[0]}v%{customdata[1]}</b><br>Archive: %{x:.3f} MB<br>Time: %{y:.2f} s<extra></extra>",
    ),
    row=1, col=2,
)
fig.add_hline(y=rd.ARXIV_DELAY, line_dash="dash", line_color=rd.INK, line_width=1.2, row=1, col=2)
fig.update_layout(title="Per-version download time and dependence on archive size")
fig.update_xaxes(title_text=f"Seconds per version (last bin: {cap} s+)", row=1, col=1)
fig.update_yaxes(title_text="Versions", row=1, col=1)
fig.update_xaxes(
    type="log", title_text="Downloaded archive size (log scale)",
    tickvals=[0.001, 0.01, 0.1, 1, 10, 100], ticktext=["1 kB", "10 kB", "100 kB", "1 MB", "10 MB", "100 MB"],
    row=1, col=2,
)
fig.update_yaxes(
    type="log", title_text="Seconds per version (log scale)",
    tickvals=[0.1, 1, 3, 10, 30, 100, 300], ticktext=["0.1", "1", "3", "10", "30", "100", "300"],
    row=1, col=2,
)
rd.show(fig, height=370)
rd.figure_caption(f"Figure 4.2. Left: distribution of the time per version. Right: time against the size of the "
                  f"downloaded archive for a random sample of {len(sample):,} versions; the dashed line is the "
                  f"{rd.ARXIV_DELAY:g} s request interval.")

classes = pd.cut(versions["downloaded_bytes"] / rd.MB, [-1, 1, 5, 20, 100, np.inf],
                 labels=["up to 1 MB", "1–5 MB", "5–20 MB", "20–100 MB", "over 100 MB"])
t43 = versions.groupby(classes, observed=True).agg(versions=("id", "size"), median=("seconds", "median"),
                                                   mean=("seconds", "mean"), total=("seconds", "sum")).reset_index()
t43["share_versions"] = 100 * t43["versions"] / len(versions)
t43["share_time"] = 100 * t43["total"] / seconds.sum()
t43 = t43[["downloaded_bytes", "versions", "share_versions", "median", "mean", "share_time"]]
t43.columns = ["Downloaded archive", "Versions", "Share of versions", "Median time (s)", "Mean time (s)",
               "Share of the total time"]
rd.table(t43, "Table 4.3. Time per version by size of the downloaded archive.",
         {"Share of versions": "{:.1f}%", "Median time (s)": "{:.2f}", "Mean time (s)": "{:.2f}",
          "Share of the total time": "{:.1f}%"})

slow = versions[seconds > 10]
rate = (slow["downloaded_bytes"] / slow["seconds"] / rd.MB).median()
small = versions["downloaded_bytes"] <= rd.MB
rd.md(
    f"The median version takes **{seconds.median():.2f} s**, just above the {rd.ARXIV_DELAY:g} s that arXiv "
    f"requires between two requests: for a small archive the request interval is the whole cost. Versions faster "
    f"than {rd.ARXIV_DELAY:g} s are those that follow a long transfer and get their slot at once. The mean is "
    f"**{seconds.mean():.2f} s** because of the large archives: versions over 5 MB are "
    f"{(versions['downloaded_bytes'] > 5 * rd.MB).mean():.1%} of the versions but "
    f"{versions.loc[versions['downloaded_bytes'] > 5 * rd.MB, 'seconds'].sum() / seconds.sum():.1%} of the time. "
    f"For the versions that take more than 10 s the transfer runs at a median of **{rate:.2f} MB/s**, so their "
    f"time is the download speed of `export.arxiv.org`, not processing. The archives are downloaded whole "
    f"although {1 - versions['after_bytes'].sum() / versions['before_bytes'].sum():.1%} of their content is "
    f"dropped afterwards: arXiv serves the source as one package, so the figures cannot be skipped at download "
    f"time.")

## 4.3 Memory and disk footprint

### RAM

In [ ]:
series = pd.DataFrame(perf.get("ram_timeseries", []), columns=["seconds", "mb"])
gap = series["seconds"].diff() > 600           # do not join the samples of two different sessions
fig = go.Figure(go.Scatter(
    x=hours(series["seconds"]),
    y=series["mb"].mask(gap),
    mode="lines",
    name="resident set size",
    line=dict(color=rd.BLUE, width=1.6),
    hovertemplate="Hour %{x:.2f}<br>RAM: <b>%{y:,.1f} MB</b><extra></extra>",
))
fig.add_hline(
    y=bench["ram_avg_mb"], line_dash="dash", line_color=rd.INK, line_width=1.2,
    annotation_text=f"average {bench['ram_avg_mb']:,.0f} MB", annotation_position="top left",
)
fig.add_hline(
    y=bench["ram_peak_mb"], line_dash="dot", line_color=rd.ORANGE, line_width=1.2,
    annotation_text=f"peak {bench['ram_peak_mb']:,.0f} MB", annotation_position="top left",
)
fig.update_layout(title="Memory of the crawler process (Google Colab CPU)")
fig.update_xaxes(title_text="Hours since the first session started", range=[0, hours((main_end - t0).total_seconds()) * 1.03])
fig.update_yaxes(title_text="RAM (MB)", range=[0, bench["ram_peak_mb"] * 1.15])
rd.show(fig, height=340)
rd.figure_caption(f"Figure 4.3. Resident memory of the notebook process over the crawl ({len(series):,} points, "
                  "downsampled from the 5 s samples). Gaps are the time between sessions.")

peak_session = int(sessions["ram_peak_mb"].idxmax()) + 1
downloading = sessions[sessions["download_seconds"].notna()]
rd.md(
    f"**Peak RAM is {bench['ram_peak_mb']:,.0f} MB and the average {bench['ram_avg_mb']:,.0f} MB** (average of the "
    f"sessions weighted by their duration), far below the roughly 12 GB of a Colab CPU runtime. Memory does not "
    f"grow with the corpus: an archive is streamed to the local disk in 1 MB blocks and unpacked from there, and "
    f"only the text files that are kept are read. The peak belongs to session {peak_session}, an interrupted "
    f"session in which no download stage was recorded; in the sessions that downloaded, the peak stays between "
    f"{downloading['ram_peak_mb'].min():,.0f} and {downloading['ram_peak_mb'].max():,.0f} MB.")

### Disk

In [ ]:
unpacked = versions["before_bytes"].sum()
downloaded = versions["downloaded_bytes"].sum()
kept = versions["after_bytes"].sum()
values = {
    "sources unpacked, all files (never stored)": unpacked,
    "archives downloaded (deleted after unpacking)": downloaded,
    "maximum disk used": bench["max_disk_bytes"],
    "final output (data/raw)": bench["final_output_bytes"],
    "… of which .tex and .bib files": kept,
}
labels_v = list(values)
gb_v = [v / rd.GB for v in values.values()]
colors = [rd.AXIS, rd.AXIS, rd.BLUE, rd.BLUE, rd.BLUE]

fig = go.Figure(go.Bar(
    y=labels_v,
    x=gb_v,
    orientation="h",
    marker_color=colors,
    text=[f"{g:,.2f} GB" for g in gb_v],
    textposition="outside",
    customdata=np.array(list(values.values()))[:, None],
    hovertemplate="<b>%{y}</b><br>%{x:,.2f} GB (%{customdata[0]:,.0f} bytes)<extra></extra>",
))
fig.update_layout(title="Data handled (grey) and disk actually used (blue)")
fig.update_yaxes(autorange="reversed", showgrid=False)
fig.update_xaxes(title_text="GB", range=[0, unpacked / rd.GB * 1.18])
rd.show(fig, height=320, margin=dict(l=270, r=25, t=55, b=45))
rd.figure_caption("Figure 4.4. Volume that passed through the scraper compared with the disk it used.")

tmp_peak = sessions["local_temp_peak_bytes"].max()
rd.md(
    f"The scraper downloaded **{downloaded / rd.GB:,.1f} GB** of archives, which unpack to "
    f"{unpacked / rd.GB:,.1f} GB, but never stores them: each archive is unpacked on the runtime's local disk, "
    f"only the `.tex`/`.bib` files are copied to Drive, and the archive is deleted. At most "
    f"`2 × DOWNLOAD_WORKERS` archives wait for a worker, so the local disk never held more than "
    f"**{tmp_peak / rd.MB:,.0f} MB** of them. The **maximum disk usage is {bench['max_disk_bytes'] / rd.GB:.2f} GB** "
    f"(project folder on Drive plus the peak in `/tmp`) and the **final output is "
    f"{bench['final_output_bytes'] / rd.GB:.2f} GB**: {kept / rd.GB:.2f} GB of `.tex`/`.bib` files, the rest being "
    f"the `metadata.json` and `references.json` of the {bench['papers_on_arxiv']:,} papers. The disk grows "
    f"linearly with the corpus and stays far below the disk of a Colab instance. These sizes are computed from "
    f"the receipts and the files written (`MEASURE_DISK = False`), which avoids walking tens of thousands of "
    f"folders on Drive at the end of every session; they therefore include the versions whose folders were found "
    f"missing on Drive afterwards (section 3.1).")

## 4.4 Summary of the performance metrics

The table lists the metrics required by the statement in one place. The values are those of
`logs/benchmark.json`, computed by `compute_benchmark()` in `src/05_run_pipeline.ipynb` over all 30 sessions
covering all 20,000 assigned IDs on Google Colab CPU.

In [ ]:
rows = [
    ("Scraping", "Papers scraped successfully", f"{bench['papers_scraped']:,} of {bench['ids_assigned']:,}"),
    ("Scraping", "Overall success rate", f"{bench['success_rate_of_assigned_pct']:.2f}%"),
    ("Scraping", "Source versions processed", f"{bench['versions_processed']:,} of {bench['versions_total']:,}"),
    ("Scraping", "Average paper size before removing figures", f"{bench['avg_paper_bytes_before']:,.0f} bytes"),
    ("Scraping", "Average paper size after removing figures", f"{bench['avg_paper_bytes_after']:,.0f} bytes"),
    ("Scraping", "Average references per paper", f"{bench['avg_references_per_paper']:.2f}"),
    ("Scraping", "Reference metadata success rate", f"{bench['reference_success_rate_pct']:.2f}%"),
    ("Running time", "Total wall time, end to end", f"{bench['wall_seconds_total']:,.0f} s ({hms(bench['wall_seconds_total'])})"),
    ("Running time", "Average time per paper, end to end", f"{bench['avg_seconds_per_paper_end_to_end']:.2f} s"),
    ("Running time", "Median time to process a paper", f"{bench['paper_download_seconds_median']:.2f} s"),
    ("Running time", "Total entry discovery time", f"{bench['discovery_seconds']:,.0f} s ({bench['discovery_seconds'] / 60:.1f} min)"),
    ("Memory", "Maximum RAM used", f"{bench['ram_peak_mb']:,.0f} MB"),
    ("Memory", "Average RAM consumption", f"{bench['ram_avg_mb']:,.0f} MB"),
    ("Disk", "Maximum disk storage required", f"{bench['max_disk_bytes'] / rd.MB:,.0f} MB ({bench['max_disk_bytes'] / rd.GB:.2f} GB)"),
    ("Disk", "Final output size", f"{bench['final_output_bytes'] / rd.MB:,.0f} MB ({bench['final_output_bytes'] / rd.GB:.2f} GB)"),
]
rd.table(pd.DataFrame(rows, columns=["Group", "Metric", "Value"]),
         f"Table 4.4. Milestone 1 metrics over {bench['sessions']} Colab sessions (computed {bench['computed_at'][:10]}).")

## 4.5 What limits the speed

arXiv allows one connection and one request every 3 s, so a compliant scraper cannot process more than
**1,200 versions per hour**, whatever its code does. The figure compares the crawl with that ceiling.

In [ ]:
ceiling = 3600 / rd.ARXIV_DELAY
window = ordered[ordered["finished_at"] <= main_end]
per_bin = window.groupby(hours((window["finished_at"] - t0).dt.total_seconds()) // 3).size() / 3
fig = go.Figure(go.Bar(
    x=per_bin.index * 3 + 1.5,
    y=per_bin.values,
    width=2.6,
    marker_color=rd.BLUE,
    customdata=np.stack([per_bin.index * 3, per_bin.index * 3 + 3], axis=-1),
    hovertemplate="Hours %{customdata[0]:.0f}–%{customdata[1]:.0f}<br><b>%{y:,.0f}</b> versions/h<extra></extra>",
))
fig.add_hline(
    y=ceiling, line_dash="dash", line_color=rd.INK, line_width=1.2,
    annotation_text=f"ceiling: {ceiling:,.0f} versions/h (1 req / {rd.ARXIV_DELAY:g} s)",
    annotation_position="top left",
)
fig.update_layout(title="Throughput of the crawl (3-hour windows)")
fig.update_xaxes(title_text="Hours since the first session started", range=[0, hours((main_end - t0).total_seconds()) * 1.03], showgrid=False)
fig.update_yaxes(title_text="Versions per hour", range=[0, ceiling * 1.15])
rd.show(fig, height=340)
rd.figure_caption("Figure 4.5. Versions processed per hour in 3-hour windows, against the ceiling set by the "
                  "arXiv request interval.")

by_day = events.groupby([events["at"].dt.strftime("%Y-%m-%d"), "error_kind"]).size().unstack(fill_value=0)
by_day = by_day.reset_index().rename(columns={"at": "Day (UTC)"})
by_day.columns.name = None
rd.table(by_day, "Table 4.5. Failed and refused source requests per day, by error kind.")

active_rate = len(versions) / hours(active_seconds)
best = per_bin.max()
floor_seconds = len(versions) * rd.ARXIV_DELAY
rd.md(
    f"**1. The request interval is the floor.** {len(versions):,} versions at {rd.ARXIV_DELAY:g} s each are "
    f"{hms(floor_seconds)} that no implementation can avoid. The sum of the measured version times is "
    f"{hms(versions['seconds'].sum())}, {versions['seconds'].sum() / floor_seconds:.2f}× that floor, and while it "
    f"was active the downloader averaged **{active_rate:,.0f} versions per hour** ({active_rate / ceiling:.0%} of "
    f"the ceiling; best 3-hour window {best:,.0f} per hour).\n\n"
    f"**2. Above the floor, the cost is network transfer.** Table 4.3 shows where the extra time goes: the large "
    f"archives, downloaded at about {rate:.2f} MB/s. Nothing on our side can shorten a transfer on a single "
    f"connection; what the scraper does is avoid *repeating* it, by resuming a cut transfer with an HTTP `Range` "
    f"request instead of starting over.\n\n"
    f"**3. CPU and Drive I/O are not on the critical path.** Unpacking, filtering and the writes to Google Drive "
    f"run in two worker threads during the wait for the next request slot, and the references are fetched on a "
    f"separate thread against a different service. The small archives confirm it: "
    f"{small.mean():.1%} of the versions are under 1 MB and their median time is "
    f"{versions.loc[small, 'seconds'].median():.2f} s, the request interval and nothing more.\n\n"
    f"**4. The real losses were interruptions and rate limiting, not speed.** The calendar time "
    f"({hms(calendar_seconds)}) is far longer than the active time ({hms(active_seconds)}): the crawl waited "
    f"between Colab sessions, and Figure 4.5 has windows with almost no progress although sessions were running. "
    f"Over the crawl arXiv answered HTTP 429 to {int((events['error_kind'] == 'HTTP_429').sum()):,} logged requests "
    f"and cut {int((events['error_kind'] == 'TRANSFER_CUT').sum()):,} transfers (Table 4.5). Colab IPs are shared, so a "
    f"runtime can be refused through no fault of its own request rate; the pipeline now tests the IP with a few "
    f"requests at start-up and asks for a new runtime instead of crawling at a fraction of the pace. Because "
    f"every version has a receipt and every stage resumes from Drive, an interruption costs the time until the "
    f"restart, not the work already done.\n\n"
    f"**5. Discovery and references are negligible.** Batching makes them so: 100 IDs per arXiv API call "
    f"({bench['discovery_seconds'] / 60:.1f} minutes in total) and 100 papers per Semantic Scholar request. With "
    f"one request per paper, as in our first version, the references alone would have taken longer than the "
    f"downloads (see [2.4 Reference metadata](references.md)).")